# 02 - ACE Training
Train the localized ACE model on the dataset and log metrics.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.ace_wrapper import ACEWrapper
from src.trainer import BenchmarkTrainer

In [2]:
# Load Data
train_ds = MDTrajectoryDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MDTrajectoryDataset("../data/val.extxyz", cutoff=5.0)
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize ACE model (shared training setup, model-specific architecture)
model = ACEWrapper(
    num_elements=120,
    num_radial=8,
    l_max=2,
    r_cut=5.0,
    hidden_dim=32
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)
print(f"Training on device: {trainer.device}\n")

Training on device: cuda



In [ ]:
# Train + held-out test evaluation
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/ace_metrics.csv", index=False)

test_metrics = trainer.test_epoch(test_loader)
ace_test_df = pd.DataFrame([test_metrics])
ace_test_df.to_csv("../data/ace_test_metrics.csv", index=False)

# Save the trained model state
torch.save(model.state_dict(), "../data/ace_model.pth")

metrics_df.head()

Epoch 000 | Time: 3.59s | Train E MAE: 22.23 meV/atom | Train F MAE: 267.31 meV/Å | Val E MAE: 22.83 meV/atom | Val F MAE: 210.02 meV/Å
Epoch 001 | Time: 1.59s | Train E MAE: 14.61 meV/atom | Train F MAE: 125.10 meV/Å | Val E MAE: 8.12 meV/atom | Val F MAE: 88.20 meV/Å
Epoch 002 | Time: 0.94s | Train E MAE: 5.06 meV/atom | Train F MAE: 77.68 meV/Å | Val E MAE: 2.87 meV/atom | Val F MAE: 73.37 meV/Å
Epoch 003 | Time: 0.98s | Train E MAE: 2.00 meV/atom | Train F MAE: 68.49 meV/Å | Val E MAE: 1.62 meV/atom | Val F MAE: 64.77 meV/Å
Epoch 004 | Time: 0.96s | Train E MAE: 4.65 meV/atom | Train F MAE: 58.74 meV/Å | Val E MAE: 12.80 meV/atom | Val F MAE: 53.69 meV/Å
Epoch 005 | Time: 0.99s | Train E MAE: 7.19 meV/atom | Train F MAE: 47.63 meV/Å | Val E MAE: 1.76 meV/atom | Val F MAE: 41.88 meV/Å
Epoch 006 | Time: 0.96s | Train E MAE: 15.43 meV/atom | Train F MAE: 35.82 meV/Å | Val E MAE: 14.83 meV/atom | Val F MAE: 31.53 meV/Å
Epoch 007 | Time: 0.95s | Train E MAE: 5.23 meV/atom | Train F MAE:

,epoch,loss,e_mae,f_mae,time,val_loss,val_e_mae,val_f_mae
0,0,11.298026,22.228195,267.306641,3.589244,6.915127,22.826621,210.022692
1,1,2.705244,14.605416,125.100720,1.592876,1.224838,8.124456,88.202082
2,2,0.953675,5.056711,77.684279,0.944411,0.840363,2.874837,73.373258
3,3,0.742078,2.001354,68.490683,0.978762,0.655564,1.616838,64.774687
4,4,0.550172,4.650301,58.740648,0.958679,0.461238,12.796836,53.685583


In [5]:
# Check model size (number of parameters)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 6,474
Trainable Parameters: 6,465
